# Size factors and the mean floor for NB-GLM

`nb_glm_test` puts every cell's counts on a common scale with a size factor: the model's mean for gene *g* in cell *i* is `s_i * exp(beta_g + effect)`. crispyx offers three ways to compute `s_i` (`size_factor_method`; the [Differential expression](../differential_expression.rst) guide's Size factors section describes them):

- `"libsize"` (default): each cell's total count;
- `"geomeans"`: causarray's median of ratios to the controls' geometric means, over genes with a control mean of at least 2;
- `"deseq2"`: DESeq2's median of ratios on the genes counted in every control.

It also floors fitted means at `min_mu` while fitting, as DESeq2 does (0.5 for bulk counts, 1e-6 for single cells in DESeq2's own guidance); by default crispyx picks the floor from how sparse the counts are.

This notebook is the evidence for those defaults. It builds screens with **known truth** from real control cells -- fake perturbations changed in known ways -- and measures how often each choice calls an unchanged (perturbation, gene) pair significant, on single cells and on pseudo-bulk units:

1. **Semi-synthetic screens**: a pure depth difference, weak perturbations, and strong ones that shift the cells' composition, by gene density.
2. **The mean floor**: 0.5 against 1e-6 on the same screens.
3. **Real cells**: fake perturbations of random control cells and of the deepest ones, with two median-of-ratios variants for reference (the `"sparse"` method crispyx used before 0.2.3, and DESeq2's `poscounts`).

Everything here is single seeds on two datasets' control cells; a benchmark across the screens is planned.

The tables are cached in `tutorial_outputs/size_factors/` and recomputed only when crispyx's fitting code (`glm.py`, `_kernels.py`, `_size_factors.py`, `de.py`; edits to comments and docstrings do not count), the parameters or the inputs change; set `FORCE = True` below to recompute regardless. From scratch the notebook takes about 15 minutes on 32 cores.

In [1]:
import ast
import hashlib
import json
import os
import sys
import time
import warnings
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Paths stay relative to this notebook's directory so that the paths crispyx
# echoes in its output are the same on every machine.
ROOT = Path('../..')
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('TQDM_DISABLE', '1')
warnings.filterwarnings('ignore', message='IProgress not found')
import crispyx as cx
from crispyx._kernels import _compute_row_medians_csr
from crispyx.de import _global_size_factors

DATA_DIR = ROOT / 'data' / 'size_factors'
OUTPUT_DIR = Path('tutorial_outputs') / 'size_factors'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_JOBS = min(32, os.cpu_count())
METHODS = ('libsize', 'geomeans', 'deseq2')
pd.set_option('display.precision', 3)

# Each table below is cached in OUTPUT_DIR with a key over the crispyx source
# its numbers depend on, its parameters and its inputs, and recomputed only
# when one of them changes (or FORCE is set).
FORCE = False
RESULT_SOURCES = ('glm.py', '_kernels.py', '_size_factors.py', 'de.py')


def _without_docstrings(tree):
    for node in ast.walk(tree):
        body = getattr(node, 'body', None)
        if (isinstance(node, (ast.Module, ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef))
                and body and isinstance(body[0], ast.Expr)
                and isinstance(body[0].value, ast.Constant) and isinstance(body[0].value.value, str)):
            node.body = body[1:] or [ast.Pass()]
    return tree


def source_key():
    # The parsed code without docstrings, so editing a comment, a docstring
    # or the layout keeps the cache.
    digest = hashlib.sha256()
    for name in RESULT_SOURCES:
        tree = _without_docstrings(ast.parse((ROOT / 'src' / 'crispyx' / name).read_text()))
        digest.update(ast.dump(tree).encode())
    return digest.hexdigest()[:16]


def cached(name, params, compute):
    """The table `name`, from the cache if its key matches, else computed and saved."""
    key = hashlib.sha256(json.dumps({'source': source_key(), **params}, sort_keys=True, default=str).encode())
    key = key.hexdigest()[:16]
    table, meta = OUTPUT_DIR / f'{name}.csv', OUTPUT_DIR / f'{name}.json'
    if not FORCE and table.exists() and meta.exists() and json.loads(meta.read_text())['key'] == key:
        print(f'{name}: from the cache (key {key})')
        return pd.read_csv(table)
    t0 = time.time()
    frame = compute()
    frame.to_csv(table, index=False)
    meta.write_text(json.dumps({'key': key, 'params': params}, indent=1, default=str))
    print(f'{name}: computed in {time.time() - t0:.0f} s (key {key})')
    return frame


print(f'crispyx {cx.__version__}, {N_JOBS} cores')

crispyx 0.2.2, 32 cores


## Data

Two screens' QC-filtered raw counts, as the benchmark writes them (`results/<dataset>/preprocessing/crispyx_qc_filtered.h5ad`), placed in `data/size_factors/`:

- **Adamson** (CRISPRi, 65,282 cells, 7,629 controls), as is;
- **Huang-HCT116** (genome-wide CRISPRi, 3.4 million cells): 20,000 controls and 20 perturbations, drawn with seed 0 by the cell below if the subset is missing and the full file is there. Every control in a comparison enters through crispyx's control summary, so the number of controls matters for the size factors (Huang has 6 genes counted in every one of these controls, Adamson 29), not for run time.

In [2]:
def subset_screen(src, out, n_targets=20, max_controls=20000, seed=0, control='control'):
    """A seeded sample of controls and perturbations, CSR, sorted by perturbation."""
    backed = ad.read_h5ad(src, backed='r')
    labels = backed.obs['perturbation'].astype(str).to_numpy()
    rng = np.random.default_rng(seed)
    chosen = rng.choice(np.setdiff1d(np.unique(labels), [control]), n_targets, replace=False)
    controls = np.flatnonzero(labels == control)
    controls = np.sort(rng.choice(controls, min(max_controls, controls.size), replace=False))
    rows = np.union1d(controls, np.flatnonzero(np.isin(labels, chosen)))
    X = sp.vstack([sp.csr_matrix(backed.X[rows[i:i + 50_000]]) for i in range(0, rows.size, 50_000)], format='csr')
    obs = pd.DataFrame({'perturbation': labels[rows]}, index=backed.obs_names[rows])
    backed.file.close()
    unsorted = out.with_suffix('.unsorted.h5ad')
    ad.AnnData(X, obs=obs, var=pd.DataFrame(index=backed.var_names)).write(unsorted)
    cx.data.sort_by_perturbation(unsorted, perturbation_column='perturbation', control_label=control, output_path=out)
    unsorted.unlink()


DATASETS = {'Adamson': DATA_DIR / 'Adamson.h5ad', 'Huang-HCT116': DATA_DIR / 'Huang-HCT116_ctrl20k_pert20.h5ad'}
huang_full = DATA_DIR / 'Huang-HCT116.h5ad'
if not DATASETS['Huang-HCT116'].exists() and huang_full.exists():
    subset_screen(huang_full, DATASETS['Huang-HCT116'])
INPUTS = {}  # each input's identity, part of the cache keys
for name, path in DATASETS.items():
    backed = ad.read_h5ad(path, backed='r')
    labels = backed.obs['perturbation'].astype(str)
    INPUTS[name] = dict(bytes=path.stat().st_size, shape=backed.shape)
    print(f'{name}: {backed.n_obs:,} cells x {backed.n_vars:,} genes, {(labels == "control").sum():,} controls')
    backed.file.close()

Adamson: 65,282 cells x 19,568 genes, 7,629 controls
Huang-HCT116: 24,350 cells x 36,326 genes, 20,000 controls


## 1. Semi-synthetic screens

Fake perturbations are random groups of 150 control cells -- real cells with no biology of their own -- changed in known ways; the remaining controls stay controls. Five per scenario:

| scenario | change | what it tests |
|---|---|---|
| depth | every count thinned to 60 % (binomial) | a pure depth difference, no gene changed: size factors must absorb it |
| weak | 30 genes (control mean >= 0.1) 1.4-2-fold, half up, half down | the common case |
| strong up | 300 of the 1,000 most expressed genes up 3-fold | a composition shift: total RNA rises |
| strong down | the same genes down 3-fold | total RNA falls |

Down-regulation thins counts binomially (exact); up-regulation adds Poisson counts with the right mean. Every other (perturbation, gene) pair is a true null. Genes are grouped by control mean: sparse (< 0.1), moderate (0.1-1), dense (>= 1). For **pseudo-bulk**, every label's cells are summed in random units of 25 (6 per fake perturbation).

In [3]:
SCENARIOS = ('depth', 'weak', 'strong_up', 'strong_down')


def semi_synthetic(path, name, per=5, size=150, unit=25, seed=0, control='control'):
    """Fake perturbations of control cells with known changes; single-cell and pseudo-bulk files."""
    rng = np.random.default_rng(seed)
    src = ad.read_h5ad(path)
    ctrl = np.flatnonzero(src.obs['perturbation'].astype(str).to_numpy() == control)
    X = sp.csr_matrix(src.X[ctrl], dtype=np.float64)
    genes = src.var_names.astype(str)
    mean = np.asarray(X.mean(axis=0)).ravel()
    stratum = np.where(mean < 0.1, 'sparse', np.where(mean < 1.0, 'moderate', 'dense'))
    top = np.argsort(mean)[::-1][:1000]
    order = rng.permutation(X.shape[0])
    labels = np.full(X.shape[0], control, dtype=object)
    truth, pos = {}, 0
    for scen in SCENARIOS:
        for k in range(per):
            pert, rows = f'{scen}_{k}', order[pos:pos + size]
            pos += size
            labels[rows] = pert
            Y = X[rows].toarray()
            fold = np.ones(X.shape[1])
            if scen == 'depth':
                Y = rng.binomial(Y.astype(np.int64), 0.6).astype(np.float64)
            elif scen == 'weak':
                g = rng.choice(np.flatnonzero(mean >= 0.1), 30, replace=False)
                fold[g] = 2.0 ** (np.where(np.arange(30) < 15, 1, -1) * rng.uniform(np.log2(1.4), 1.0, 30))
            else:
                fold[rng.choice(top, 300, replace=False)] = 3.0 if scen == 'strong_up' else 1 / 3
            down, up = fold < 1, fold > 1
            Y[:, down] = rng.binomial(Y[:, down].astype(np.int64), fold[down][None, :])
            Y[:, up] = Y[:, up] + rng.poisson((fold[up][None, :] - 1.0) * Y[:, up])
            X[rows] = sp.csr_matrix(Y)
            truth[pert] = fold != 1
    X.eliminate_zeros()
    sc_path = OUTPUT_DIR / f'{name}_single_cell.h5ad'
    ad.AnnData(X, obs=pd.DataFrame({'perturbation': labels}, index=[f'c{i}' for i in range(X.shape[0])]),
               var=pd.DataFrame(index=genes)).write(sc_path)
    units, unit_labels = [], []
    for lab in [control, *truth]:
        rows = rng.permutation(np.flatnonzero(labels == lab))
        for i in range(rows.size // unit):
            units.append(rows[i * unit:(i + 1) * unit])
            unit_labels.append(lab)
    agg = sp.csr_matrix((np.ones(sum(map(len, units))),
                         (np.repeat(np.arange(len(units)), [len(u) for u in units]), np.concatenate(units))),
                        shape=(len(units), X.shape[0]))
    pb_path = OUTPUT_DIR / f'{name}_pseudo_bulk.h5ad'
    ad.AnnData(sp.csr_matrix(agg @ X), obs=pd.DataFrame({'perturbation': unit_labels},
                                                        index=[f'u{i}' for i in range(len(units))]),
               var=pd.DataFrame(index=genes)).write(pb_path)
    return {'single cells': sc_path, 'pseudo-bulk': pb_path}, truth, stratum


def evaluate(path, truth, stratum, method, min_mu=None, control='control'):
    """Per scenario and gene stratum: false-positive rate (p < 0.05), mean null z,
    false hits (q < 0.1) per perturbation, power (q < 0.1)."""
    out = OUTPUT_DIR / f'{path.stem}_{method}_{min_mu}.h5ad'
    res = cx.nb_glm_test(path, perturbation_column='perturbation', control_label=control,
                         size_factor_method=method, min_mu=min_mu,
                         n_jobs=N_JOBS, verbose=False, force=True, output_path=out)
    r = ad.read_h5ad(res.result_path)
    rows = list(r.obs_names.astype(str))
    z, p, q = (np.asarray(r.layers[k]) for k in ('z_score', 'pvalue', 'pvalue_adj'))
    records = []
    for scen in SCENARIOS:
        idx = [i for i, lab in enumerate(rows) if lab.startswith(scen)]
        tr = np.vstack([truth[rows[i]] for i in idx])
        tested = np.isfinite(p[idx])
        for st in ('all', 'sparse', 'moderate', 'dense'):
            cols = np.ones(tr.shape[1], bool) if st == 'all' else stratum == st
            null, alt = tested & ~tr & cols, tested & tr & cols
            records.append(dict(
                method=method, min_mu=float(r.uns['min_mu']), scenario=scen, genes=st,
                fpr=100 * np.mean(p[idx][null] < 0.05) if null.any() else np.nan,
                mean_null_z=np.mean(z[idx][null]) if null.any() else np.nan,
                false_hits=((q[idx] < 0.1) & null).sum() / len(idx),
                power=np.mean(q[idx][alt] < 0.1) if alt.any() else np.nan))
    return records


_screens = {}


def screen(name):
    """The semi-synthetic screen of a dataset, built on first use."""
    if name not in _screens:
        _screens[name] = semi_synthetic(DATASETS[name], name)
    return _screens[name]


SCREEN_PARAMS = dict(inputs=INPUTS, per=5, size=150, unit=25, seed=0)

In [4]:
def default_runs_table():
    rows = []
    for name in DATASETS:
        files, truth, stratum = screen(name)
        for level, path in files.items():
            for method in METHODS:
                rows += [dict(dataset=name, level=level, **rec) for rec in evaluate(path, truth, stratum, method)]
    return pd.DataFrame(rows)


default_runs = cached('semi_synthetic_defaults', dict(SCREEN_PARAMS, methods=METHODS), default_runs_table)
default_runs.groupby(['dataset', 'level']).min_mu.first().to_frame('min_mu chosen')

semi_synthetic_defaults: from the cache (key 37d2d66615d96d9e)


min_mu chosen
dataset      level                      
Adamson      pseudo-bulk       5.000e-01
             single cells      1.000e-06
Huang-HCT116 pseudo-bulk       1.000e-06
             single cells      1.000e-06

The floor was chosen from the counts: single cells are mostly zeros (1e-6); Adamson's 25-cell units are not (0.5), but Huang's still are, because most of its 36,000 genes are very sparse (1e-6). Section 2 compares the two floors directly.

**False-positive rate** (% of null pairs at p < 0.05; 5 % expected), all genes and well-counted genes (control mean >= 1):

In [5]:
def table(frame, value, genes='all'):
    sub = frame[frame.genes == genes]
    return sub.pivot_table(index=['level', 'scenario'], columns=['method', 'dataset'], values=value).reindex(
        columns=METHODS, level=0).reindex(index=SCENARIOS, level=1)


print('all genes'); display(table(default_runs, 'fpr').round(1))
print('well-counted genes (control mean >= 1)'); display(table(default_runs, 'fpr', 'dense').round(0))

all genes


method                   libsize              geomeans               deseq2  \
dataset                  Adamson Huang-HCT116  Adamson Huang-HCT116 Adamson   
level        scenario                                                         
pseudo-bulk  depth           3.6          5.3      3.6          5.3     3.7   
             weak            3.4          4.1      3.4          4.1     3.4   
             strong_up      45.5         43.8      3.3          4.4     3.3   
             strong_down    20.8         17.5      3.3          4.4     3.4   
single cells depth           5.6          5.8      7.1          8.1     5.5   
             weak            5.1          4.8      5.0          4.7     5.1   
             strong_up      47.4         44.6     20.3         15.4    10.2   
             strong_down    23.9         18.8     22.7         19.7    17.6   

method                                 
dataset                  Huang-HCT116  
level        scenario                  
pseudo-bulk  depth                5.4  
             weak                 4.1  
             strong_up            4.4  
             strong_down          4.5  
single cells depth                5.9  
             weak                 4.5  
             strong_up           25.1  
             strong_down         34.9

well-counted genes (control mean >= 1)


method                   libsize              geomeans               deseq2  \
dataset                  Adamson Huang-HCT116  Adamson Huang-HCT116 Adamson   
level        scenario                                                         
pseudo-bulk  depth           5.0          5.0      5.0          6.0     5.0   
             weak            5.0          4.0      5.0          3.0     5.0   
             strong_up      99.0         98.0      5.0          5.0     5.0   
             strong_down    67.0         46.0      5.0          4.0     5.0   
single cells depth           5.0          6.0     14.0         16.0     5.0   
             weak            6.0          4.0      6.0          4.0     6.0   
             strong_up     100.0         98.0     67.0         46.0    30.0   
             strong_down    68.0         49.0     65.0         52.0    45.0   

method                                 
dataset                  Huang-HCT116  
level        scenario                  
pseudo-bulk  depth                6.0  
             weak                 3.0  
             strong_up            5.0  
             strong_down          4.0  
single cells depth                7.0  
             weak                 4.0  
             strong_up           56.0  
             strong_down         74.0

**False hits** (q < 0.1) per perturbation, and **power** on the changed pairs:

In [6]:
display(table(default_runs, 'false_hits').round(0))
display(table(default_runs, 'power').dropna(how='all').round(2))

method                   libsize              geomeans               deseq2  \
dataset                  Adamson Huang-HCT116  Adamson Huang-HCT116 Adamson   
level        scenario                                                         
pseudo-bulk  depth           0.0          7.0      0.0          7.0     0.0   
             weak            2.0          5.0      1.0          5.0     1.0   
             strong_up    5839.0       7780.0     15.0         35.0    16.0   
             strong_down  1742.0       1426.0     22.0         37.0    23.0   
single cells depth           2.0          5.0     15.0         35.0     3.0   
             weak            5.0          7.0      6.0          8.0     6.0   
             strong_up    6138.0       7953.0   1535.0       1014.0   312.0   
             strong_down  2109.0       1614.0   1906.0       1872.0  1414.0   

method                                 
dataset                  Huang-HCT116  
level        scenario                  
pseudo-bulk  depth                7.0  
             weak                 5.0  
             strong_up           36.0  
             strong_down         38.0  
single cells depth               10.0  
             weak                 6.0  
             strong_up         3656.0  
             strong_down       5553.0

method                   libsize              geomeans               deseq2  \
dataset                  Adamson Huang-HCT116  Adamson Huang-HCT116 Adamson   
level        scenario                                                         
pseudo-bulk  weak           0.53         0.51     0.53         0.53    0.53   
             strong_up      1.00         1.00     1.00         1.00    1.00   
             strong_down    1.00         1.00     1.00         1.00    1.00   
single cells weak           0.53         0.53     0.53         0.52    0.53   
             strong_up      1.00         1.00     1.00         1.00    1.00   
             strong_down    1.00         1.00     1.00         1.00    1.00   

method                                 
dataset                  Huang-HCT116  
level        scenario                  
pseudo-bulk  weak                0.52  
             strong_up           1.00  
             strong_down         1.00  
single cells weak                0.51  
             strong_up           1.00  
             strong_down         1.00

What the tables show:

- **Library size** follows depth best: under pure depth thinning its tests are calibrated in single cells, while `"geomeans"` over-calls well-counted genes (a median of ratios over a cell's *non-zero* counts is pushed up when thinning turns counts into zeros). It fails under **composition shifts**, in single cells and pseudo-bulk alike, because the total moves with the changed genes.
- **Geometric means** and **DESeq2's ratio** resist composition shifts on **pseudo-bulk** completely: pseudo-bulk counts are large, so a median of ratios is both depth-accurate and robust.
- In **single cells no method stays calibrated under a strong composition shift**: a median over counts of 1-5 resists a shift far less. The ratio method is the least affected when many genes are counted in every control (Adamson, 29) and the most when few are (Huang, 6, some among the changed ones).
- Under **weak perturbations** all three agree, at both levels.

## 2. The mean floor

NB-GLM holds fitted means at least `min_mu` while fitting. DESeq2's 0.5 suits bulk counts; on sparse counts many genes have means below one, and the floor binds more often in the shallower group. The same screens with each floor fixed (library size; the other size factors move the same way):

In [7]:
def floor_runs_table():
    rows = []
    for name in DATASETS:
        files, truth, stratum = screen(name)
        for level, path in files.items():
            for min_mu in (0.5, 1e-6):
                rows += [dict(dataset=name, level=level, **rec)
                         for rec in evaluate(path, truth, stratum, 'libsize', min_mu=min_mu)]
    return pd.DataFrame(rows)


floor_runs = cached('semi_synthetic_floors', dict(SCREEN_PARAMS, method='libsize', floors=(0.5, 1e-6)),
                    floor_runs_table)
sub = floor_runs[floor_runs.scenario.isin(['depth', 'weak'])]
summary = pd.concat({
    'FPR %, all genes': sub[sub.genes == 'all'].pivot_table(index=['level', 'scenario'], columns=['dataset', 'min_mu'], values='fpr'),
    'mean null z, moderate genes': sub[sub.genes == 'moderate'].pivot_table(index=['level', 'scenario'], columns=['dataset', 'min_mu'], values='mean_null_z'),
    'power (weak)': sub[(sub.genes == 'all') & (sub.scenario == 'weak')].pivot_table(index=['level', 'scenario'], columns=['dataset', 'min_mu'], values='power'),
})
summary.round(3)

semi_synthetic_floors: from the cache (key 58eeccfda8172961)


dataset                                             Adamson            \
min_mu                                            1.000e-06 5.000e-01   
                            level        scenario                       
FPR %, all genes            pseudo-bulk  depth        5.005     3.623   
                                         weak         4.311     3.437   
                            single cells depth        5.616     4.886   
                                         weak         5.058     1.700   
mean null z, moderate genes pseudo-bulk  depth       -0.028    -0.029   
                                         weak        -0.001    -0.004   
                            single cells depth       -0.027     1.197   
                                         weak         0.000     0.002   
power (weak)                pseudo-bulk  weak         0.527     0.527   
                            single cells weak         0.533     0.440   

dataset                                           Huang-HCT116            
min_mu                                               1.000e-06 5.000e-01  
                            level        scenario                         
FPR %, all genes            pseudo-bulk  depth           5.314     3.781  
                                         weak            4.090     2.998  
                            single cells depth           5.758     3.985  
                                         weak            4.811     1.254  
mean null z, moderate genes pseudo-bulk  depth          -0.046    -0.046  
                                         weak           -0.006    -0.006  
                            single cells depth          -0.054     1.082  
                                         weak            0.004    -0.007  
power (weak)                pseudo-bulk  weak            0.513     0.513  
                            single cells weak            0.533     0.420

On single cells, 0.5 makes tests conservative (far fewer than 5 % of null pairs below 0.05, less power) and biases moderately expressed genes when the groups differ in depth (mean null z well above 0); 1e-6 is calibrated. On pseudo-bulk both floors are acceptable, 0.5 the more conservative. Hence the default, `min_mu=None`: **1e-6 when at least half of a sample of the counts are zero, 0.5 otherwise** -- a rule about sparsity rather than the kind of data, recorded in the result's `uns["min_mu"]`. Pass a number to override it, e.g. 0.5 to match a default PyDESeq2 or DESeq2 run.

## 3. Real cells: random and deep fake perturbations

Thinning is an idealised depth difference. Here fake perturbations are real groups of 150 control cells, either random or drawn from the deepest 30 % of controls -- a real depth difference, which also brings some biology along (deeper cells tend to be larger), so no method is perfectly calibrated on them; the **mean null z** on well-counted genes measures normalisation bias. Two median-of-ratios variants are added for reference, passed as `size_factors=`: crispyx's `"sparse"` before 0.2.3 (non-zero geometric means over all cells) and DESeq2's `poscounts` against the controls. 40 real perturbations (all 20 on the Huang subset) show how much the choice moves real results.

In [8]:
def reference_size_factors(X, is_ctrl):
    """'sparse' (pre-0.2.3) and DESeq2 'poscounts' against the controls, geometric mean 1."""
    def medians(geo):
        sf = _compute_row_medians_csr(X.data, X.indices.astype(np.int64), X.indptr.astype(np.int64), geo, X.shape[0])
        sf[~(np.isfinite(sf) & (sf > 0))] = np.nanmedian(sf)
        return sf / np.exp(np.mean(np.log(sf)))

    out = {}
    for name, cells_, denom in (('sparse (pre-0.2.3)', np.ones(X.shape[0], bool), 'nnz'), ('poscounts', is_ctrl, 'n')):
        csc = X[cells_].tocsc()
        nnz = np.diff(csc.indptr)
        logged = csc.copy()
        logged.data = np.log(logged.data)
        log_sum = np.asarray(logged.sum(axis=0)).ravel()
        geo = np.zeros(X.shape[1])
        ok = nnz > 0
        geo[ok] = np.exp(log_sum[ok] / (nnz[ok] if denom == 'nnz' else csc.shape[0]))
        out[name] = medians(geo)
    return out


def real_fakes(path, name, n_real=40, n_fake=20, size=150, seed=0, control='control'):
    rng = np.random.default_rng(seed)
    adata = ad.read_h5ad(path)
    X = sp.csr_matrix(adata.X, dtype=np.float64)
    labels = adata.obs['perturbation'].astype(str).to_numpy()
    real = np.setdiff1d(np.unique(labels), [control])
    ctrl = np.flatnonzero(labels == control)
    depth = np.asarray(X[ctrl].sum(axis=1)).ravel()
    deep = rng.permutation(ctrl[depth >= np.quantile(depth, 0.7)])
    n = min(n_fake, deep.size // size, ctrl.size // (6 * size))  # keep >= 2/3 of the controls
    new = labels.copy()
    for i in range(n):
        new[deep[i * size:(i + 1) * size]] = f'DEEP{i:02d}'
    rest = rng.permutation(np.flatnonzero(new == control))
    for i in range(n):
        new[rest[i * size:(i + 1) * size]] = f'FAKE{i:02d}'
    chosen = sorted(rng.choice(real, min(n_real, real.size), replace=False))
    chosen += [f'FAKE{i:02d}' for i in range(n)] + [f'DEEP{i:02d}' for i in range(n)]
    adata.obs = pd.DataFrame({'perturbation': new}, index=adata.obs_names)
    fake_path = OUTPUT_DIR / f'{name}_real_fakes.h5ad'
    adata.write(fake_path)
    is_ctrl = new == control
    expressed = np.asarray(X[is_ctrl].mean(axis=0)).ravel() >= 1.0
    sfs = {m: _global_size_factors(fake_path, m, 2048, new, control) for m in METHODS}
    sfs.update(reference_size_factors(X, is_ctrl))
    records = []
    for method, sf in sfs.items():
        res = cx.nb_glm_test(fake_path, perturbation_column='perturbation', control_label=control,
                             perturbations=chosen, size_factors=sf,
                             n_jobs=N_JOBS, verbose=False, force=True,
                             output_path=OUTPUT_DIR / f'{name}_real_fakes_{method.split()[0]}.h5ad')
        r = ad.read_h5ad(res.result_path)
        rows = list(r.obs_names.astype(str))
        z, p, q = (np.asarray(r.layers[k]) for k in ('z_score', 'pvalue', 'pvalue_adj'))
        lfc = np.asarray(r.X)
        rec = dict(dataset=name, method=method, min_mu=float(r.uns['min_mu']))
        for kind, prefix in (('random', 'FAKE'), ('deep', 'DEEP')):
            idx = [i for i, lab in enumerate(rows) if lab.startswith(prefix)]
            ok = np.isfinite(p[idx]) & expressed
            rec[f'{kind}: FPR %'] = 100 * np.mean(p[idx][ok] < 0.05)
            rec[f'{kind}: mean null z'] = np.mean(z[idx][ok])
            rec[f'{kind}: false hits'] = (q[idx] < 0.1).sum(axis=1).mean()
        real_i = [i for i, lab in enumerate(rows) if not lab.startswith(('FAKE', 'DEEP'))]
        rec['real: hits'] = (q[real_i] < 0.1).sum(axis=1).mean()
        rec['real: |median LFC|'] = np.median(np.abs([np.nanmedian(lfc[i]) for i in real_i]))
        records.append(rec)
    fake_path.unlink()
    print(f'{name}: {n} random and {n} deep fake perturbations, {len(chosen) - 2 * n} real')
    return records


real_runs = cached(
    'real_fakes', dict(inputs=INPUTS, n_real=40, n_fake=20, size=150, seed=0, methods=METHODS),
    lambda: pd.DataFrame([rec for name, path in DATASETS.items() for rec in real_fakes(path, name)]))
real_runs.set_index(['dataset', 'min_mu', 'method']).round(2)

real_fakes: from the cache (key 62533c5a40130df0)


random: FPR %  random: mean null z  \
dataset      min_mu    method                                                   
Adamson      1.000e-06 libsize                      5.73                -0.03   
                       geomeans                     5.81                -0.03   
                       deseq2                       6.60                -0.05   
                       sparse (pre-0.2.3)           5.56                -0.26   
                       poscounts                    5.26                -0.24   
Huang-HCT116 1.000e-06 libsize                      5.14                -0.03   
                       geomeans                     5.47                -0.10   
                       deseq2                       6.32                -0.19   
                       sparse (pre-0.2.3)           6.40                -0.12   
                       poscounts                    6.37                -0.11   

                                           random: false hits  deep: FPR %  \
dataset      min_mu    method                                                
Adamson      1.000e-06 libsize                           1.88        22.15   
                       geomeans                          1.25        23.20   
                       deseq2                            1.75        23.25   
                       sparse (pre-0.2.3)                1.25        94.90   
                       poscounts                         1.50        94.34   
Huang-HCT116 1.000e-06 libsize                           4.55        13.97   
                       geomeans                          4.95        24.36   
                       deseq2                           10.05        25.00   
                       sparse (pre-0.2.3)                3.65        99.35   
                       poscounts                         3.75        98.83   

                                           deep: mean null z  \
dataset      min_mu    method                                  
Adamson      1.000e-06 libsize                         -0.08   
                       geomeans                         0.29   
                       deseq2                          -0.46   
                       sparse (pre-0.2.3)               5.27   
                       poscounts                        5.09   
Huang-HCT116 1.000e-06 libsize                          0.11   
                       geomeans                         0.97   
                       deseq2                           0.95   
                       sparse (pre-0.2.3)               5.62   
                       poscounts                        5.16   

                                           deep: false hits  real: hits  \
dataset      min_mu    method                                             
Adamson      1.000e-06 libsize                       209.75     2362.18   
                       geomeans                      246.88     2312.57   
                       deseq2                        232.75     2898.25   
                       sparse (pre-0.2.3)           5398.12     2761.78   
                       poscounts                    5269.75     2771.12   
Huang-HCT116 1.000e-06 libsize                        41.85       50.40   
                       geomeans                      403.00       54.45   
                       deseq2                        695.90       91.70   
                       sparse (pre-0.2.3)           9940.30      308.80   
                       poscounts                    9369.00      247.00   

                                           real: |median LFC|  
dataset      min_mu    method                                  
Adamson      1.000e-06 libsize                           0.02  
                       geomeans                          0.02  
                       deseq2                            0.08  
                       sparse (pre-0.2.3)                0.07  
                       poscounts                         0.07  
Huang-HCT116 1.

On random fakes every method is close to calibrated on well-counted genes. On deep fakes, a median of ratios over **every** expressed gene (`"sparse"`, `poscounts`) calls nearly every well-counted gene up (mean null z around +5, thousands of false hits): most single-cell counts are 1 or 2, so those ratios capture only part of a depth difference. That is why `"sparse"` was removed in 0.2.3 and why `"geomeans"` keeps only genes with a control mean of at least 2. Library size has the least depth bias. On real perturbations the choice moves hit counts several-fold.

## Summary

| | single cells | pseudo-bulk |
|---|---|---|
| pure depth difference | `"libsize"` and `"deseq2"` calibrated; `"geomeans"` over-calls well-counted genes | all calibrated |
| weak perturbations | all calibrated | all calibrated |
| strong composition shift | none calibrated; `"libsize"` worst | `"geomeans"` and `"deseq2"` calibrated; `"libsize"` fails |
| mean floor | 1e-6 | 0.5 (or 1e-6 for very sparse units) |

- **Single cells**: keep the defaults (`"libsize"`, the floor chosen from the counts).
- **Pseudo-bulk and bulk**: `size_factor_method="geomeans"` or `"deseq2"`.
- **Perturbations with large effects on highly expressed genes**: aggregate to pseudo-bulk units (several guides or donors per perturbation) and use `"geomeans"`, or at least compare such a perturbation's median log fold change across size factors -- a whole-perturbation shift with `"libsize"` that `"geomeans"` lacks is a composition shift.
- **Comparing with PyDESeq2 or DESeq2**: give them the same size factors and floor (`dds.obs["size_factors"]` and `min_mu`; `sizeFactors(dds) <- ...` and `minmu`).

In [9]:
# The intermediate h5ad files are large; keep only the cached tables.
for path in OUTPUT_DIR.glob('*.h5ad'):
    path.unlink()
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

['real_fakes.csv', 'real_fakes.json', 'semi_synthetic_defaults.csv', 'semi_synthetic_defaults.json', 'semi_synthetic_floors.csv', 'semi_synthetic_floors.json']
